# Glacier evolution using OGGM + whatif

In [ ]:
from pathlib import Path
import xarray as xr 
import pandas as pd


import plotly.express as px
import plotly.graph_objects as go
from   plotly.subplots import make_subplots
from sklearn.preprocessing import MinMaxScaler

from src.utils import load_config
from src.plotting import base_layout, scenario_colors, geo_style, add_region_boundary_ruler, add_panel_label, save_figure
cfg = load_config()

scenarios_names = [s["name"] for s in cfg["scenarios"]]

## Data

In [ ]:
# volume data (Glacier-specific data)
glacier_specific_vol = pd.read_csv("data/glacier/glacier_metadata_evolution.csv", index_col=0)
glacier_specific_vol["cenlat"] = glacier_specific_vol["cenlat"].round(0)
glacier_specific_vol["cenlon"] = glacier_specific_vol["cenlon"].round(0)
glacier_specific_vol = glacier_specific_vol.groupby(['cenlat', 'cenlon']).sum()
glacier_specific_vol = glacier_specific_vol[["rgi_area_km2", "volume_2020_historical", "volume_2100_15C", "volume_2100_28C", "volume_2100_40C"]]
for col in ["volume_2100_15C", "volume_2100_28C", "volume_2100_40C"]:
    glacier_specific_vol[col] = 1 - (glacier_specific_vol[col] / glacier_specific_vol["volume_2020_historical"])
glacier_specific_vol = glacier_specific_vol.dropna().reset_index()

# volume regional data 
hist_data   = xr.open_dataset("data/glacier/glacier_evolution_2000_2024.nc").volume
future_data = xr.open_dataset("data/glacier/glacier_evolution_2025_2099.nc").volume
future_data["time"] = future_data["time"] + 1 # TODO check
hist_ref  = hist_data.sel(time=2020)
hist_data = hist_data / hist_ref
future_data = future_data / hist_ref

# Volume observations (GLAMBIE)
glambie_data = pd.read_csv("data/glacier/GLAMBIE_southern_andes.csv").set_index('end_dates')
glambie_data['volume_delta_km3'] = glambie_data['combined_gt'] / 0.85
initial_volume_km3 = glacier_specific_vol.volume_2020_historical.sum() / 1e9
glambie_data['volume_total'] = initial_volume_km3 + glambie_data['volume_delta_km3'].cumsum()
glambie_data['volume_norm'] = glambie_data['volume_total'] / glambie_data.loc[2020, 'volume_total']

# glacier runoff regional data (+ what-if)
glacier_hist_data = xr.open_dataset("data/glacier/run_output_historical.nc", engine="h5netcdf").glacier_melt
ids               = pd.Index(glacier_hist_data.gauge_id.values.astype(str))
glacier_hist_data = glacier_hist_data.sel(gauge_id=ids[ids.str.match(r"^\d")])
glacier_hist_data = glacier_hist_data.sum(dim='gauge_id')

files = list(Path("data/glacier/").glob("run_output_historical_*.nc"))
whatif_data       = xr.open_mfdataset(files, concat_dim = "warming_level",  combine='nested' )
whatif_data       = whatif_data.sel(gauge_id=ids[ids.str.match(r"^\d")])
whatif_data       = whatif_data.sum(dim='gauge_id').glacier_melt.load()
whatif_data       = whatif_data.assign_coords(warming_level=[f.name.split('_')[3] for f in files])

# Apply 15-day moving average
glacier_hist_data = glacier_hist_data.rolling(date=15, center=True).mean()
glacier_hist_data = glacier_hist_data.groupby(glacier_hist_data.date.dt.dayofyear).mean()
whatif_data = whatif_data.rolling(date=15, center=True).mean()
whatif_data = whatif_data.groupby(whatif_data.date.dt.dayofyear).mean()

future_data_mean = future_data.groupby('warming_level').mean().assign_coords(warming_level=scenarios_names)
future_data_sd   = future_data.groupby('warming_level').std().assign_coords(warming_level=scenarios_names)
whatif_data_mean = whatif_data.groupby('warming_level').mean().assign_coords(warming_level=scenarios_names)
whatif_data_sd   = whatif_data.groupby('warming_level').std().assign_coords(warming_level=scenarios_names)

In [ ]:
colors = scenario_colors(cfg)
scaler = MinMaxScaler(feature_range=(8, 35))

# Create subplots
fig = make_subplots(rows=2, cols=2, horizontal_spacing=0.08, vertical_spacing=0.07, column_widths=[0.4, 0.6], 
                    specs=[[{"type": "scattergeo", "rowspan": 2}, {"type": "xy"}],
                           [None, {"type": "xy"}]])

colorscale = ["#F8E070", "#F4B354", "#EC8439", "#E05020", "#C84232", "#AF3540", "#96274B", "#7C1B55"]

fig.add_trace(
    go.Scattergeo(
        lon=glacier_specific_vol['cenlon'],
        lat=glacier_specific_vol['cenlat'],
        mode='markers',
        marker=dict(
            size=scaler.fit_transform(glacier_specific_vol[['rgi_area_km2']]).flatten(),
            colorscale=colorscale,
            color=glacier_specific_vol['volume_2100_28C'] * 100,  # Convert to percentage
            cmin=50,
            cmax=100,
            opacity=0.8,
            colorbar=dict(
                len=0.30,
                x=0.08,
                y=0.62,
                thickness=17,
                ticks="outside",
                tickfont=dict(size=13),
                ticksuffix="%",
                title=dict(text="Glacier loss<br>(2020–2100)", font=dict(size=16)),
            ),
            line=dict(width=0.1, color='black')
        ),
        showlegend=False
    ),
    row=1, col=1
)

# Plot observed data from glambie_data
fig.add_trace(go.Scatter(
    x=glambie_data.index,
    y=glambie_data['volume_norm'] * 100,
    mode='lines',
    name='Observed (GlaMBIE)',
    line=dict(color='grey', width=2, dash='dot')
), row=1, col=2)

# First subplot - Historical data
fig.add_trace(go.Scatter(
    x=hist_data.time, 
    y=hist_data.values * 100, 
    mode='lines',
    name='Simulated (OGGM)',
    legend="legend1",
    line=dict(color='black', width=2)
), row=1, col=2)

fig.add_trace(go.Scatter(
    x=glacier_hist_data.dayofyear, 
    y=glacier_hist_data.values, 
    mode='lines',
    showlegend=False,
    legend="legend1",
    line=dict(color='black', width=2)
), row=2, col=2)

# Plot future scenarios with confidence intervals for both subplots
for i, warming_level in enumerate(scenarios_names):
    # First subplot data
    mean_data_vol = future_data_mean.sel(warming_level=warming_level)
    std_data_vol = future_data_sd.sel(warming_level=warming_level)
    
    # Second subplot data
    mean_data_melt = whatif_data_mean.sel(warming_level=warming_level)
    std_data_melt = whatif_data_sd.sel(warming_level=warming_level)
    
    # Convert hex to rgba for fillcolor
    hex_color = colors[i].lstrip('#')
    rgb = tuple(int(hex_color[j:j+2], 16) for j in (0, 2, 4))
    rgba_color = f'rgba({rgb[0]}, {rgb[1]}, {rgb[2]}, 0.2)'
    
    # First subplot - Volume confidence interval
    upper_vol = (mean_data_vol + std_data_vol) * 100
    lower_vol = (mean_data_vol - std_data_vol) * 100
    
    fig.add_trace(go.Scatter(
        x=list(mean_data_vol.time) + list(mean_data_vol.time)[::-1],
        y=list(upper_vol.values) + list(lower_vol.values)[::-1],
        fill='toself',
        fillcolor=rgba_color,
        line=dict(color='rgba(255,255,255,0)'),
        showlegend=False,
        hoverinfo="skip",
        name=warming_level,
    ), row=1, col=2)
    
    # First subplot - Volume mean line
    fig.add_trace(go.Scatter(
        x=mean_data_vol.time, 
        y=mean_data_vol.values * 100,  # Convert to percentage
        mode='lines',
        name=warming_level,
        legend = "legend2",
        line=dict(color=colors[i], width=2)
    ), row=1, col=2)
    
    # Second subplot - Melt confidence interval
    upper_melt = mean_data_melt + std_data_melt
    lower_melt = mean_data_melt - std_data_melt

    fig.add_trace(go.Scatter(
        x=list(upper_melt.dayofyear) + list(upper_melt.dayofyear)[::-1],
        y=list(upper_melt.values) + list(lower_melt.values)[::-1],
        fill='toself',
        fillcolor=rgba_color,
        line=dict(color='rgba(255,255,255,0)'),
        showlegend=False,
        hoverinfo="skip",
        name=f'+{warming_level} warming (±1σ)'
    ), row=2, col=2)
    
    # Second subplot - Melt mean line
    fig.add_trace(go.Scatter(
        x=mean_data_melt.dayofyear, 
        y=mean_data_melt.values,
        mode='lines',
        name=warming_level,
        line=dict(color=colors[i], width=2),
        showlegend=False
    ), row=2, col=2)


# Update x and y axes
fig.update_xaxes(range=[2000, 2100], row=1, col=2)
fig.update_xaxes(
    title_standoff=0,
    tickvals=[15,46,74,105,135,166,196,227,258,288,319,349],
    ticktext=["J", "F", "M", "A", "M", "J", "J", "A", "S", "O", "N", "D"],
    range=[1, 365],
    row=2, col=2
)

fig.update_yaxes(title_text='Glacier volume rel. to 2020 (%)', title_standoff=0, range=[0, 120], row=1, col=2)
fig.update_yaxes(title_text='Glacier melt (m<sup>3</sup>s<sup>-1</sup>)', title_standoff=0, dtick = 2000, range = [0, 8000], row=2, col=2)

# Add subplot annotations
add_panel_label(fig, "a)", x=0.36, y=0.995)
add_panel_label(fig, "b)", x=0.99, y=0.995)
add_panel_label(fig, "c)", x=0.99, y=0.45)

fig.add_annotation(
    text="Historical climate (2000–2024)<br>replayed starting from projected<br>2100 glacier geometry",
    x=75,
    y=3100,
    xref="x2",
    yref="y2",
    ax=200,
    ay=5000,
    axref="x2",
    ayref="y2",
    showarrow=True,
    arrowhead=5,
    arrowsize=1,
    arrowwidth=1.5,
    arrowcolor="#757d88",
    font=dict(size=16),
    align="left",
)


# Update geography settings for each subplot
fig.update_geos(**geo_style([-82, -62]), row=1, col=1)

# Add region-boundary ruler on the left of the map
add_region_boundary_ruler(
    fig, cfg["regions"], boundary_lats=[-7.2, -20, -30, -35, -45, -52, -55.8], borderpad=2,
)

# Update layout
fig.update_layout(**base_layout(
    width=900,
    height=800,
    font=dict(size=15),
    margin=dict(l=20, r=20, b=40, t=20),
    legend=dict(yanchor="top", title = "Historical:", y=0.65, x=0.46, bgcolor="rgba(255,255,255,0)"),
    legend2=dict(yanchor="top", title = "Scenarios:", y=0.68, x=0.70, bgcolor="rgba(255,255,255,0)"),
))

save_figure(fig, "figures/fig_glacier_evolution.svg", width=900, height=800)
fig.show()